# Notebook 03 - Statistical Analysis

**Requirement 7:** Statistical analysis with p-values.

This notebook covers:
- Descriptive statistics per city
- Pearson correlation analysis (r + p-value)
- Welch's independent t-test (price differences between cities)
- Simple linear regression (price ~ floor area)

In [ ]:
import pandas as pd, numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt, seaborn as sns
from itertools import combinations
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

In [ ]:
df = pd.read_csv('../data/inserate_bereinigt.csv')
print(f'Dataset: {len(df)} listings from {df["stadt"].nunique()} cities')
df[['preis_chf','flaeche_m2','zimmer_anzahl','preis_pro_m2']].describe()

## 1. Descriptive Statistics per City

In [ ]:
descriptive = df.groupby('stadt')['preis_chf'].agg(
    n='count', mean='mean', median='median', std='std', min='min', max='max'
).round(0).sort_values('mean', ascending=False)
print('Price statistics per city (CHF/month):')
descriptive

## 2. Pearson Correlation Analysis

**H0:** No linear correlation (r = 0)

Tests the linear relationship between numeric variables.
- **r**: Pearson coefficient (-1 to +1)
- **p-value**: probability of observing this correlation by chance
- Significance: *** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant

In [ ]:
num_cols = ['preis_chf','flaeche_m2','zimmer_anzahl','preis_pro_m2']
df_num = df[num_cols].dropna()
print('Pearson Correlation Analysis:')
print('='*65)
print(f'{"Variable 1":20} {"Variable 2":20} {"r":>8} {"p-value":>12} Sig.')
print('-'*65)
for col1, col2 in combinations(num_cols, 2):
    r, p = stats.pearsonr(df_num[col1], df_num[col2])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    print(f'{col1:20} {col2:20} {r:8.4f} {p:12.2e} {sig}')
print('\nSignificance: *** p<0.001  ** p<0.01  * p<0.05  n.s. = not significant')

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
corr_matrix = df_num.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap='RdYlGn', vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Pearson Correlation Matrix', pad=15)
plt.tight_layout()
plt.savefig('../data/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/correlation_matrix.png')

## 3. Welch's t-test: Zurich vs. Other Cities

**H0:** No significant price difference between Zurich and the compared city (alpha = 0.05)

Welch's t-test is used because sample variances differ between cities (more robust than Student's t-test).

In [ ]:
zurich = df[df['stadt']=='Zuerich']['preis_chf'].dropna()
print(f'Welch t-test: Zurich (n={len(zurich)}, avg CHF {zurich.mean():.0f}) vs. other cities')
print('H0: No significant price difference (alpha = 0.05)')
print('='*70)
print(f'{"City":15}{"n":>5}{"Avg price":>10}{"t-stat":>12}{"p-value":>12} Decision')
print('-'*70)
for city in sorted([c for c in df['stadt'].unique() if c != 'Zuerich' and pd.notna(c)]):
    g = df[df['stadt']==city]['preis_chf'].dropna()
    if len(g) < 5: continue
    t, p = stats.ttest_ind(zurich, g, equal_var=False)  # Welch's t-test
    d = 'Reject H0' if p < 0.05 else 'Retain H0'
    print(f'{city:15}{len(g):5}{g.mean():10.0f}{t:12.4f}{p:12.4e} {d}')
print('Method: Welchs t-test (unequal variances, independent samples)')

## 4. Linear Regression: Price ~ Floor Area

Tests how well floor area predicts rental price.
- **Slope:** price increase per additional m2
- **R2:** proportion of price variance explained by the model
- **p-value:** significance of the regression

In [ ]:
x, y = df_num['flaeche_m2'].values, df_num['preis_chf'].values
slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
print('Linear Regression: Price ~ Floor Area')
print(f'  Model:    Price = {slope:.2f} x Area (m2) + {intercept:.2f}')
print(f'  R2:       {r_val**2:.4f} ({r_val**2*100:.1f}% of variance explained)')
print(f'  p-value:  {p_val:.2e}')
print(f'  -> Each additional m2 costs on average CHF {slope:.2f} more in rent')

fig, ax = plt.subplots(figsize=(9,6))
for c in df['stadt'].dropna().unique():
    s = df[df['stadt']==c]
    ax.scatter(s['flaeche_m2'], s['preis_chf'], alpha=0.5, s=30, label=c)
xl = np.linspace(x.min(), x.max(), 200)
ax.plot(xl, slope*xl+intercept, 'k-', lw=2.5, label=f'Regression (R2={r_val**2:.3f})')
ax.set_xlabel('Floor Area (m2)', fontsize=12)
ax.set_ylabel('Rental Price (CHF/month)', fontsize=12)
ax.set_title('Rental Price vs. Floor Area - Swiss Cities', fontsize=13)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/regression_price_area.png', dpi=150, bbox_inches='tight')
plt.show()